# Static Maps (matplotlib)

`viz.render()` produces interactive Leaflet maps for notebooks and
dashboards. `viz.render_static()` produces matplotlib figures for
reports, papers, grant applications, and batch thumbnail generation —
any context where a static PNG or PDF is needed.

**What's covered**
1. Basic `render_static()` call
2. All four layers: isochrone, network, POI scatter, heatmap
3. Layer subsets
4. `figsize` and `dpi` for publication-quality output
5. Saving to PNG, PDF, and SVG
6. Generating a multi-panel comparison figure
7. Basemap via contextily (optional)


## Setup

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt

from idd_travel_time.models import TravelTimeRequest
from idd_travel_time.constants import TransportMode, TimeContext, VizLayer
from idd_travel_time.isochrone import compute
from idd_travel_time import viz

OUTPUT_DIR = Path(tempfile.mkdtemp())

# Compute one result to use throughout this notebook
request = TravelTimeRequest.from_place(
    "Kenyatta National Hospital, Nairobi",
    TransportMode.WALKING,
    duration_minutes=20,
)
result = compute(request)
print(f"Computed: {result.router_name}")


## 1 · Basic static render

In [ ]:
fig = viz.render_static(result)
plt.show()


## 2 · Layer subsets

Pass an explicit `layers=` list to show only what you need.
Useful when the network layer clutters a report figure.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, layers, title in [
    (axes[0], [VizLayer.ISOCHRONE],                    "Isochrone only"),
    (axes[1], [VizLayer.ISOCHRONE, VizLayer.HEATMAP],  "Isochrone + Heatmap"),
]:
    # render_static returns its own figure — we re-use the axes manually
    # by passing the result through the internal layer functions directly
    pass   # shown properly in the individual cell below

plt.tight_layout()
plt.close()

# Better: render separately and compare side-by-side
fig1 = viz.render_static(result, layers=[VizLayer.ISOCHRONE], figsize=(7, 7))
fig2 = viz.render_static(result, layers=[VizLayer.ISOCHRONE, VizLayer.HEATMAP], figsize=(7, 7))
plt.show()


## 3 · `figsize` and `dpi` for publication quality

In [ ]:
# Journal-quality: 3.5-inch single column at 300 dpi
fig_journal = viz.render_static(result, figsize=(3.5, 3.5), dpi=300)
fig_journal.axes[0].set_title("Walking access (20 min) · KNH, Nairobi", fontsize=8)
plt.show()
print(f"Figure size: {fig_journal.get_size_inches()} inches at {fig_journal.get_dpi()} dpi")


## 4 · Saving to PNG, PDF, SVG

In [ ]:
png_path = viz.save_static(result, OUTPUT_DIR / "nairobi_walking.png", dpi=150)
pdf_path = viz.save_static(result, OUTPUT_DIR / "nairobi_walking.pdf", dpi=300)
svg_path = viz.save_static(result, OUTPUT_DIR / "nairobi_walking.svg")

print(f"PNG: {png_path}  ({png_path.stat().st_size // 1024} KB)")
print(f"PDF: {pdf_path}  ({pdf_path.stat().st_size // 1024} KB)")
print(f"SVG: {svg_path}  ({svg_path.stat().st_size // 1024} KB)")


## 5 · Multi-panel comparison figure

Generate one figure with panels for multiple transport modes —
useful for a methods section or a city profile.


In [ ]:
from idd_travel_time.models import TransitProfile

# Compute three isochrones from the same origin
modes_config = [
    ("Walking 20 min",   TransportMode.WALKING,  20, None, TimeContext.OFF_PEAK),
    ("Cycling 20 min",   TransportMode.CYCLING,  20, None, TimeContext.OFF_PEAK),
    ("Driving 15 min",   TransportMode.DRIVING,  15, None, TimeContext.RUSH_HOUR),
]

modal_results = []
for label, mode, dur, _, ctx in modes_config:
    req = TravelTimeRequest(
        origin_lat=request.origin_lat,
        origin_lon=request.origin_lon,
        mode=mode,
        duration_minutes=dur,
        time_context=ctx,
    )
    modal_results.append((label, compute(req)))

# Build a 1×3 panel figure
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for ax, (label, res) in zip(axes, modal_results):
    # Render to a temporary figure and copy the axes contents
    tmp_fig = viz.render_static(
        res,
        layers=[VizLayer.ISOCHRONE, VizLayer.HEATMAP],
        figsize=(6, 7),
        dpi=100,
    )
    # Copy the image rendered by render_static into our panel
    tmp_ax = tmp_fig.axes[0]
    for artist in tmp_ax.get_children():
        try:
            artist_copy = artist
        except Exception:
            pass
    plt.close(tmp_fig)

    # Simpler: just render each panel directly
    pass

plt.close()

# Cleanest multi-panel approach: render and save each, then assemble in the report
for label, res in modal_results:
    out = OUTPUT_DIR / f"panel_{label.replace(' ', '_').lower()}.png"
    viz.save_static(res, out, layers=[VizLayer.ISOCHRONE, VizLayer.HEATMAP], dpi=150)
    print(f"Saved: {out.name}")


## 6 · Batch thumbnail generation

For a global run, generate a small thumbnail for every result in a
results directory — useful for quick visual QA.


In [ ]:
from idd_travel_time.io import read_result

THUMB_DIR = OUTPUT_DIR / "thumbnails"
THUMB_DIR.mkdir(exist_ok=True)

# Use the results computed in this notebook as an example
for res in modal_results:
    label, result_item = res
    thumb_path = THUMB_DIR / f"{label.replace(' ', '_').lower()}.png"
    viz.save_static(
        result_item,
        thumb_path,
        layers=[VizLayer.ISOCHRONE],
        figsize=(4, 4),
        dpi=72,           # small thumbnails — low dpi is fine
    )

print(f"Thumbnails written to {THUMB_DIR}")
for f in sorted(THUMB_DIR.iterdir()):
    print(f"  {f.name}  {f.stat().st_size // 1024} KB")


## 7 · Basemap via contextily (optional)

`render_static` automatically adds a CartoDB Positron tiled basemap if
`contextily` is installed. Falls back gracefully to a grey background
if it's not.

```bash
pip install contextily
```

The basemap is purely cosmetic — the isochrone polygon and all other
data are still accurate without it.


In [ ]:
try:
    import contextily
    print(f"contextily {contextily.__version__} is installed — basemaps will render.")
    print("Tip: first render in a session fetches tiles; subsequent renders use the tile cache.")
except ImportError:
    print("contextily not installed — renders use a grey background.")
    print("Install with: pip install contextily")
